In [3]:
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt

In [4]:
df=pd.read_csv("C:/Users/USER/Downloads/titanic.csv")
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [5]:
# Not dropping 'Name' yet — title inside it (Mr./Mrs./Miss) hints at age & status, both linked to survival. Extract Title first, then drop.
df = df.drop(['PassengerId', 'Ticket', 'Cabin'], axis=1)
df

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...,...
886,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,13.0000,S
887,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,30.0000,S
888,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,23.4500,S
889,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,30.0000,C


In [6]:
# Extract the title (Mr, Mrs, Miss, etc.) from the Name column
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')

# Check what titles we got and how often each appears
df['Title'].value_counts()

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64

In [7]:
# Fix French/alternate title spellings
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Group all the rare titles (military, nobility, clergy, etc.) into one bucket
rare_titles = ['Dr', 'Rev', 'Major', 'Col', 'Capt', 'Countess', 'Sir', 'Lady', 'Don', 'Jonkheer']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')

# Check it worked
df['Title'].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [8]:
import math
median_age = math.floor(df['Age'].median())
df['Age'] = df['Age'].fillna(median_age)

In [9]:
# Check how many missing (NaN) values remain in each column — Age should now be 0
df.isnull().sum()

Survived    0
Pclass      0
Name        0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    2
Title       0
dtype: int64

In [10]:
df = df.drop('Name', axis=1)

In [11]:
# Fill missing Embarked values with the most common port (mode)
mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)
# Confirm no missing values remain
df.isnull().sum()


Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
Title       0
dtype: int64

In [12]:
df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)
df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,22.0,1,0,7.2500,True,False,True,False,True,False,False
1,1,1,38.0,1,0,71.2833,False,False,False,False,False,True,False
2,1,3,26.0,0,0,7.9250,False,False,True,True,False,False,False
3,1,1,35.0,1,0,53.1000,False,False,True,False,False,True,False
4,0,3,35.0,0,0,8.0500,True,False,True,False,True,False,False


In [13]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,22.0,1,0,7.2500,1,0,1,0,1,0,0
1,1,1,38.0,1,0,71.2833,0,0,0,0,0,1,0
2,1,3,26.0,0,0,7.9250,0,0,1,1,0,0,0
3,1,1,35.0,1,0,53.1000,0,0,1,0,0,1,0
4,0,3,35.0,0,0,8.0500,1,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,27.0,0,0,13.0000,1,0,1,0,0,0,1
887,1,1,19.0,0,0,30.0000,0,0,1,1,0,0,0
888,0,3,28.0,1,2,23.4500,0,0,1,1,0,0,0
889,1,1,26.0,0,0,30.0000,1,0,0,0,1,0,0


In [14]:
x=df.drop('Survived',axis=1)
y=df['Survived']

In [15]:
from sklearn.model_selection import train_test_split
x_train, x_test,y_train ,y_test=train_test_split(x,y,test_size=0.2,random_state=45)

In [27]:
x_train.shape, x_test.shape

((712, 12), (179, 12))

In [29]:
from sklearn.preprocessing import StandardScaler

# Scale features so big-range columns (Fare) don't dominate small-range ones (SibSp)
scaler = StandardScaler()

# Learn scale from train data, then apply it
x_train_scaled = scaler.fit_transform(x_train)

# Reuse train's scale on test — never fit on test (avoids data leakage)
x_test_scaled = scaler.transform(x_test)

In [30]:
from sklearn.linear_model import LogisticRegression

# Create and train the logistic regression model
model = LogisticRegression()
model.fit(x_train_scaled, y_train)

# Predict on the test set
y_pred = model.predict(x_test_scaled)

In [31]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8603351955307262


In [32]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[103  14]
 [ 11  51]]


In [33]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.88      0.89       117
           1       0.78      0.82      0.80        62

    accuracy                           0.86       179
   macro avg       0.84      0.85      0.85       179
weighted avg       0.86      0.86      0.86       179



In [34]:
from sklearn.tree import DecisionTreeClassifier

# Create and train the decision tree model (no scaling needed — trees don't care about feature scale)
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(x_train, y_train)

# Predict on the test set
y_pred_tree = tree_model.predict(x_test)

In [35]:
print(accuracy_score(y_test, y_pred_tree))
print(confusion_matrix(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

0.7597765363128491
[[87 30]
 [13 49]]
              precision    recall  f1-score   support

           0       0.87      0.74      0.80       117
           1       0.62      0.79      0.70        62

    accuracy                           0.76       179
   macro avg       0.75      0.77      0.75       179
weighted avg       0.78      0.76      0.76       179



In [36]:
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(x_train, y_train)
y_pred_tree = tree_model.predict(x_test)

print(accuracy_score(y_test, y_pred_tree))

0.8770949720670391


In [37]:
print(confusion_matrix(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

[[103  14]
 [  8  54]]
              precision    recall  f1-score   support

           0       0.93      0.88      0.90       117
           1       0.79      0.87      0.83        62

    accuracy                           0.88       179
   macro avg       0.86      0.88      0.87       179
weighted avg       0.88      0.88      0.88       179



In [38]:
# Test different tree depths to find the best balance (avoid overfitting vs underfitting)
for depth in [2, 3, 4, 5, 6, 7]:
    tree_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree_model.fit(x_train, y_train)
    acc = accuracy_score(y_test, tree_model.predict(x_test))
    print(f"max_depth={depth}: accuracy={acc:.4f}")

max_depth=2: accuracy=0.8045
max_depth=3: accuracy=0.8659
max_depth=4: accuracy=0.8771
max_depth=5: accuracy=0.8771
max_depth=6: accuracy=0.8603
max_depth=7: accuracy=0.8436


In [39]:
# Final model: depth=4 chosen — same accuracy as depth=5, but simpler tree
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(x_train, y_train)
y_pred_tree = tree_model.predict(x_test)

In [40]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest: combines many decision trees, usually more accurate & stable than one tree
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(x_train, y_train)
y_pred_rf = rf_model.predict(x_test)

print(accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

0.8044692737430168
[[94 23]
 [12 50]]
              precision    recall  f1-score   support

           0       0.89      0.80      0.84       117
           1       0.68      0.81      0.74        62

    accuracy                           0.80       179
   macro avg       0.79      0.80      0.79       179
weighted avg       0.82      0.80      0.81       179



In [41]:
for depth in [3, 4, 5, 6, 7, 8, None]:
    rf_model = RandomForestClassifier(max_depth=depth, random_state=42)
    rf_model.fit(x_train, y_train)
    acc = accuracy_score(y_test, rf_model.predict(x_test))
    print(f"max_depth={depth}: accuracy={acc:.4f}")

max_depth=3: accuracy=0.8436
max_depth=4: accuracy=0.8492
max_depth=5: accuracy=0.8603
max_depth=6: accuracy=0.8659
max_depth=7: accuracy=0.8659
max_depth=8: accuracy=0.8827
max_depth=None: accuracy=0.8045


In [42]:
from sklearn.model_selection import cross_val_score

# Cross-validation: tests the model on 5 different train/test splits instead of just one,
# to check if depth=8's accuracy is real or just a lucky split
rf_model = RandomForestClassifier(max_depth=8, random_state=42)
cv_scores = cross_val_score(rf_model, x, y, cv=5)

print("Scores for each fold:", cv_scores)
print("Average accuracy:", cv_scores.mean())

Scores for each fold: [0.82681564 0.80337079 0.87078652 0.78089888 0.85393258]
Average accuracy: 0.8271608813006089


In [43]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

log_model = LogisticRegression()
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)

print("Logistic Regression CV avg:", cross_val_score(log_model, x, y, cv=5).mean())
print("Decision Tree CV avg:", cross_val_score(tree_model, x, y, cv=5).mean())
print("Random Forest CV avg:", cv_scores.mean())

Logistic Regression CV avg: 0.812560416797439


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

Decision Tree CV avg: 0.8226539451384094
Random Forest CV avg: 0.8271608813006089


In [44]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

log_pipeline = make_pipeline(StandardScaler(), LogisticRegression())
print("Logistic Regression CV avg (scaled):", cross_val_score(log_pipeline, x, y, cv=5).mean())

Logistic Regression CV avg (scaled): 0.8260247316552632


In [69]:
df.to_csv('titanic.csv', index=False)